# CSV source-data profiler (v2, recursive)

The first run only found `financial_summary_clean.csv` because it started in `notebooks` and `gen_data` wasn't directly there. This version scans your **whole project tree**, so it won't miss `gen_data` wherever it sits.

**Set `ROOT_PATH`** (next cell) to the folder that contains `gen_data` — I've pre-filled your project root. Run it, check the printed **EXPECTED FOUND / MISSING** lines, then send back `csv_profile.json`. If anything I need is still listed as MISSING, point `ROOT_PATH` one level higher and re-run.

Needs only `pandas` + `numpy`.

In [2]:
# === CSV SOURCE-DATA PROFILER v2 (recursive) =================================
# Scans your whole project folder for *.csv so it can't miss gen_data. Writes ONE
# file: csv_profile.json. Send that back.
import os, json, math
from pathlib import Path
import pandas as pd
import numpy as np

# ---- CONFIG -----------------------------------------------------------------
# Point this at your PROJECT ROOT (it searches every subfolder, incl. gen_data).
ROOT_PATH          = "notebook/gen_data/"
OUTPUT_JSON        = "csv_profile.json"
FULL_DUMP_MAX_ROWS = 500
SAMPLE_ROWS        = 5
MAX_UNIQUE_LISTED  = 40
MAX_STR_LEN        = 4000
# Tables I most need to resolve the data issues (used only for the found/missing report):
EXPECTED = ["banks","board_minutes_extract","carbon_credits","climate_scenarios","collateral",
            "counterparties","counterparty_emissions","financial_summary",
            "climate_risk_register","employees","climate_opportunities",
            "exposures","facilities","governance","internal_carbon_price","investments","physical_risk_exposures",
            "rec_registry", "source_systems", "targets", "travel_records", "utility_invoices", "value_chain_map",
            "vehicules"]

def _clean(v):
    if v is None: return None
    if isinstance(v, (np.bool_, bool)): return bool(v)
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, (np.floating, float)):
        f = float(v)
        if math.isnan(f): return None
        if math.isinf(f): return "inf" if f > 0 else "-inf"
        return f
    if isinstance(v, pd.Timestamp):
        try: return v.isoformat()
        except Exception: return str(v)
    if isinstance(v, bytes): return v.decode("utf-8", "replace")
    if isinstance(v, str):
        return v if len(v) <= MAX_STR_LEN else v[:MAX_STR_LEN] + f"...[+{len(v)-MAX_STR_LEN} chars]"
    try:
        if pd.isna(v): return None
    except Exception: pass
    return v

def _row(row): return {str(k): _clean(v) for k, v in row.items()}

def profile_column(s):
    col = {"dtype": str(s.dtype), "non_null": int(s.notna().sum()), "nulls": int(s.isna().sum()),
           "n_unique": int(s.nunique(dropna=True))}
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
        nn = s.dropna()
        if len(nn):
            col.update(min=_clean(nn.min()), max=_clean(nn.max()),
                       mean=_clean(round(float(nn.mean()), 6)),
                       median=_clean(round(float(nn.median()), 6)),
                       sum=_clean(round(float(nn.sum()), 6)),
                       examples=[_clean(x) for x in nn.head(5).tolist()])
    else:
        vc = s.dropna().astype(str).value_counts()
        col["top_values"] = {str(k): int(v) for k, v in vc.head(MAX_UNIQUE_LISTED).items()}
        if len(vc) > MAX_UNIQUE_LISTED:
            col["top_values_note"] = f"showing {MAX_UNIQUE_LISTED} of {len(vc)} distinct values"
        col["examples"] = [_clean(x) for x in s.dropna().astype(str).head(5).tolist()]
    return col

def profile_table(df, source_rel):
    info = {"source": source_rel, "rows": int(len(df)), "columns_count": int(df.shape[1]),
            "columns": list(map(str, df.columns)),
            "column_profiles": {str(c): profile_column(df[c]) for c in df.columns}}
    for key in ("bank_id", "reporting_year", "year"):
        if key in df.columns:
            try:
                info.setdefault("key_values", {})[key] = sorted(
                    [_clean(x) for x in df[key].dropna().unique().tolist()], key=lambda z: str(z))
            except Exception: pass
    if len(df) <= FULL_DUMP_MAX_ROWS:
        info["full_dump"] = True
        info["rows_data"] = [_row(r) for _, r in df.iterrows()]
    else:
        info["full_dump"] = False
        info["sample_head"] = [_row(r) for _, r in df.head(SAMPLE_ROWS).iterrows()]
        info["sample_tail"] = [_row(r) for _, r in df.tail(SAMPLE_ROWS).iterrows()]
    return info

def read_csv_robust(f):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try: return pd.read_csv(f, encoding=enc)
        except UnicodeDecodeError: continue
    return pd.read_csv(f)

root = Path(ROOT_PATH)
search_root = root if root.exists() else Path(".")
csvs = sorted(p for p in search_root.rglob("*.csv") if ".ipynb_checkpoints" not in p.parts)

profile = {"root": str(search_root.resolve()), "n_files": len(csvs),
           "config": {"FULL_DUMP_MAX_ROWS": FULL_DUMP_MAX_ROWS}, "files": {}, "errors": {}}

for f in csvs:
    rel = str(f.relative_to(search_root))
    label = f.stem if f.stem not in profile["files"] else f"{f.parent.name}/{f.stem}"
    try:
        profile["files"][label] = profile_table(read_csv_robust(f), rel)
    except Exception as e:
        profile["errors"][label] = repr(e)

found_stems = {lbl.split("/")[-1] for lbl in profile["files"]}
profile["expected_tables_found"]   = sorted([t for t in EXPECTED if t in found_stems])
profile["expected_tables_missing"] = sorted([t for t in EXPECTED if t not in found_stems])
bank_ids = set()
for t in profile["files"].values():
    for b in (t.get("key_values", {}).get("bank_id") or []): bank_ids.add(str(b))
profile["bank_ids_across_tables"] = sorted(bank_ids)

Path(OUTPUT_JSON).write_text(json.dumps(profile, indent=2, default=str), encoding="utf-8")
print(f"Scanned {profile['root']} recursively -> {len(csvs)} CSV file(s)")
for lbl, t in profile["files"].items():
    print(f"  {lbl:34s} {t['rows']:>7d} rows x {t['columns_count']:>2d} cols  ({t['source']})")
print("\nEXPECTED FOUND  :", profile["expected_tables_found"])
print("EXPECTED MISSING:", profile["expected_tables_missing"])
print(f"\nWrote {OUTPUT_JSON} ({Path(OUTPUT_JSON).stat().st_size/1024:.1f} KB). Send this file back.")
if profile["expected_tables_missing"]:
    print("\n! Some expected tables were not found. If they live OUTSIDE ROOT_PATH, set ROOT_PATH to the folder that contains gen_data and re-run.")


Scanned C:\Users\HP\Documents\IFRS_Reporting\notebooks recursively -> 26 CSV file(s)
  financial_summary_clean                 15 rows x 18 cols  (financial_summary_clean.csv)
  banks                                    5 rows x 18 cols  (gen_data\banks.csv)
  board_minutes_extract                  154 rows x 13 cols  (gen_data\board_minutes_extract.csv)
  carbon_credits                          28 rows x 19 cols  (gen_data\carbon_credits.csv)
  climate_opportunities                   25 rows x 12 cols  (gen_data\climate_opportunities.csv)
  climate_risk_register                   80 rows x 17 cols  (gen_data\climate_risk_register.csv)
  climate_scenarios                       84 rows x 22 cols  (gen_data\climate_scenarios.csv)
  collateral                             228 rows x 19 cols  (gen_data\collateral.csv)
  counterparties                         305 rows x 20 cols  (gen_data\counterparties.csv)
  counterparty_emissions                 900 rows x 12 cols  (gen_data\counterparty_e